# Analyze Scaling Laws

Replicate plots from Andrej [miniseries_v1](https://github.com/karpathy/nanochat/discussions/420)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
RUN_DIR = "/home/user/.cache/nanorepro/runs"
SCALING_PREFIX = "scaling3_"    # only consider run directories starting with 'scaling3_'
assert os.path.exists(RUN_DIR)

In [ ]:
log_dirs = [d for d in os.listdir(RUN_DIR) if d.startswith(SCALING_PREFIX) and os.path.isdir(os.path.join(RUN_DIR, d))]
log_files = [os.path.join(RUN_DIR, log_dir, "train_log_rank0.jsonl") for log_dir in log_dirs]
log_files = sorted(log_files)
print(log_files)

In [ ]:
# Load logs
log_bpb_eval = []
for log_file in log_files:
    with open(log_file, "r") as f:
        lines = f.readlines()
        user_config = json.loads(lines[0])
        assert user_config["event"] == "user_config"
        model_config = json.loads(lines[1])
        assert model_config["event"] == "model_config"
        params_counts = json.loads(lines[2])
        assert params_counts["event"] == "params_counts"
        training_hyperparameters = json.loads(lines[3])
        assert training_hyperparameters["event"] == "training_hyperparameters"
        
        run = user_config["run"]
        target_flops = int(user_config["target_flops"])
        depth = user_config["depth"]
        scaling_params = params_counts['transformer_active'] + params_counts['lm_head']
        training_tokens = training_hyperparameters['max_steps'] * training_hyperparameters['total_batch_size']

        single_run_log_objects = [json.loads(line) for line in lines]
        single_run_bpb_evel_objects = [obj for obj in single_run_log_objects if obj['event'] == 'bpb_eval']
        single_run_extended_objects = [{
            'timestamp': obj['timestamp'],
            'event': obj['event'],
            'step': obj['step'],
            'rank': obj['rank'],
            'val/bpb': obj['val/bpb'],
            'run': run,
            'depth': depth,
            'scaling_params': scaling_params,
            'training_tokens': training_tokens,
            'target_flops': target_flops,
        } for obj in single_run_bpb_evel_objects]
        log_bpb_eval.extend(single_run_extended_objects)
        print(f"{log_file}: {len(single_run_extended_objects)} lines")

In [ ]:
df_bpb_eval = pd.DataFrame(log_bpb_eval)

In [ ]:
plot_flops = 6e18
df_filtered = df_bpb_eval[df_bpb_eval['target_flops'] == int(plot_flops)]
plt.figure(figsize=(15, 5))
for depth, g in df_filtered.groupby("depth"):
    plt.plot(g['step'], g['val/bpb'], label=f"depth={depth}")
plt.title(f"BPB Eval by Depth ({plot_flops:.0e} FLOPs)")
plt.xlabel("step")
plt.ylabel("BPB")
plt.ylim(0.75, 1.0)
plt.grid(True, which="both", lw=0.5, ls="--")
plt.legend(ncol=2, fontsize=8)
plt.savefig("analyze_scaling_laws_plot01_bpb_eval_by_depth.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
captured_optimums = []

df_tail = df_bpb_eval.sort_values('step').groupby(['run'], as_index=False).tail(1)
df_tail_sorted = df_tail.sort_values('depth')
plt.figure(figsize=(5, 5))
for iso_flops, g in df_tail_sorted.groupby("target_flops"):
    # Display raw data points
    x = g['scaling_params'].to_numpy()
    y = g['val/bpb'].to_numpy()
    plt.scatter(x, y, label=f"{iso_flops:.0e} FLOPs")
    
    # Fit BPB and display in log10 space
    x_log = np.log10(x)
    coef = np.polyfit(x_log, y, deg=2)  # returns [a, b, c]
    poly = np.poly1d(coef)
    x_fit = np.linspace(x_log.min(), x_log.max(), 100)
    y_fit = poly(x_fit)
    (line,) = plt.plot(10**x_fit, y_fit, linestyle='--')

    # Display fitted BPB minimums
    x_min_log = -coef[1] / (2 * coef[0])
    y_min = poly(x_min_log)
    plt.plot(10**x_min_log, y_min, marker='x', markersize=8, markeredgewidth=2, color=line.get_color())

    # Interpolate tokens at minimum
    tokens_min = np.interp(10**x_min_log, x, g['training_tokens'])

    # Capture for more plots
    captured_optimums.append({
        'iso_flops': iso_flops,
        'params_at_min': 10**x_min_log,
        'tokens_at_min': tokens_min,
        'ratio': tokens_min / 10**x_min_log,
        'bpb': y_min,
    })

plt.title(f"Scaling Laws IsoFLOP Curves")
plt.xlabel("Scaling Params (log scale)")
plt.ylabel("BPB")
plt.xscale("log")
plt.grid(True, which="both", lw=0.5, ls="--")
plt.legend(fontsize=8)
plt.savefig("analyze_scaling_laws_plot02_iso_flop_curves.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
print("FLOPS         Params           Tokens   Ratio      BPB")
for opt in captured_optimums:
    print(f"{opt['iso_flops']:.0e}    {opt['params_at_min']:,.0f}    {opt['tokens_at_min']:,.0f}   {opt['ratio']:.2f}   {opt['bpb']:.4f}")

This is the result I got:

```
FLOPS         Params           Tokens   Ratio      BPB
1e+18    115,399,278    1,274,356,943   11.04   0.8444
3e+18    186,137,098    2,420,658,667   13.00   0.7993
6e+18    269,470,252    3,385,658,171   12.56   0.7733
```

The compute-optimal ratio seems stable around 12-13, which corresponds to NanoChat defualt =12 in recent commits.

My conclusion is that sweep is broadly sane and valid. Having said that, optima are fitted with only six depths per FLOP budge, and minima are fairly flat around neighbouring depths, so I would urge not to overinterpret these results and treat them as approximate sanity check.

In [ ]:
df_opt = pd.DataFrame(captured_optimums)
df_opt

In [ ]:
flops = df_opt['iso_flops'].to_numpy()
params = df_opt['params_at_min'].to_numpy()
tokens = df_opt['tokens_at_min'].to_numpy()
# Fit scaling laws
params_coef = np.polyfit(np.log10(flops), np.log10(params), deg=1)  # returns [a, b]
params_poly = np.poly1d(params_coef)
tokens_coef = np.polyfit(np.log10(flops), np.log10(tokens), deg=1)  # returns [a, b]
tokens_poly = np.poly1d(tokens_coef)

# Print
print(f"log10(params) = {params_coef[0]:.4f} * log10(flops) + {params_coef[1]:.4f}")
print(f"log10(tokens) = {tokens_coef[0]:.4f} * log10(flops) + {tokens_coef[1]:.4f}")

# Equivalently
print(f"optimal params = 10**{params_coef[1]:.4f} * flops**{params_coef[0]:.4f}")
print(f"optimal tokens = 10**{tokens_coef[1]:.4f} * flops**{tokens_coef[0]:.4f}")

Result is

```
log10(params) = 0.4698 * log10(flops) + -0.3996
log10(tokens) = 0.5489 * log10(flops) + -0.7693
optimal params = 10**-0.3996 * flops**0.4698
optimal tokens = 10**-0.7693 * flops**0.5489
```

Which is close to `D ∝ C^0.5` and `N ∝ C^0.5` and confirms run sanity, broadly.

In [ ]:

x = np.linspace(np.log10(flops).min(), np.log10(flops).max(), 100)
y = params_poly(x)

plt.figure(figsize=(5, 5))
plt.title(f"Optimal Model Params")
plt.xlabel("FLOPs (log scale)")
plt.ylabel("Params (log scale)")
plt.scatter(flops, params)
plt.plot(10**x, 10**y, ls="--", label=f"N ∝ C^{params_coef[0]:.4f}")
plt.grid(True, which="both", lw=0.5, ls="--")
plt.xscale("log")
plt.yscale("log")
plt.legend(ncol=2, fontsize=8)
plt.savefig("analyze_scaling_laws_plot03_optimal_model_params.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
x = np.linspace(np.log10(flops).min(), np.log10(flops).max(), 100)
y = tokens_poly(x)

plt.figure(figsize=(5, 5))
plt.title(f"Optimal Training Tokens")
plt.xlabel("FLOPs (log scale)")
plt.ylabel("Tokens (log scale)")
plt.scatter(flops, tokens)
plt.plot(10**x, 10**y, ls="--", label=f"D ∝ C^{tokens_coef[0]:.4f}")
plt.grid(True, which="both", lw=0.5, ls="--")
plt.xscale("log")
plt.yscale("log")
plt.legend(ncol=2, fontsize=8)
plt.savefig("analyze_scaling_laws_plot04_optimal_training_tokens.png", dpi=200, bbox_inches="tight")
plt.show()